<div dir="rtl">

# الدرس 38: باي تورش: الأساسيات

دفتر الكود لهذا الدرس. شغّل الخلايا بالترتيب من الأعلى.

</div>

<div dir="rtl">

## الموترات

</div>

In [ ]:
import numpy as np
import torch

a = torch.tensor([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0]])
print(a.shape, a.dtype)
print(a @ a.T)                       # matrix product, like NumPy
print(a.mean(dim=0))                 # dim is like NumPy's axis
print(a.reshape(3, 2))
print(torch.zeros(2, 3).shape, torch.randn(4).shape)

<div dir="rtl">

### من نمباي وإليها

</div>

In [ ]:
x_np = np.array([[1.0, 2.0]])
x = torch.from_numpy(x_np)
print(x.dtype)                       # float64: NumPy's default
print(x.float().dtype)               # float32: what neural networks use
print(a.numpy().sum())               # back to NumPy

<div dir="rtl">

## التدرّج التلقائي

</div>

In [ ]:
x, y = torch.tensor(2.0), torch.tensor(7.0)
w = torch.tensor(1.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)

loss = (w * x + b - y) ** 2
loss.backward()
print("loss:", loss.item(), " dL/dw:", w.grad.item(), " dL/db:", b.grad.item())

<div dir="rtl">

### التدرّجات تتراكم!

</div>

In [ ]:
loss = (w * x + b - y) ** 2
loss.backward()
print("after a second backward, dL/dw:", w.grad.item())

w.grad.zero_(); b.grad.zero_()
print("after zeroing:", w.grad.item())

<div dir="rtl">

### نزول بالتدرّج، بالتدرّج التلقائي

</div>

In [ ]:
w = torch.tensor(1.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)
for step in range(3):
    loss = (w * x + b - y) ** 2
    loss.backward()
    with torch.no_grad():            # the update itself must not be recorded
        w -= 0.01 * w.grad
        b -= 0.01 * b.grad
    w.grad.zero_(); b.grad.zero_()
    print(f"step {step}: loss before update = {loss.item():.4f}")

<div dir="rtl">

## الشبكة كصنف

</div>

In [ ]:
from torch import nn

class MLP(nn.Module):
    def __init__(self, d_in, d_hidden, n_classes):
        super().__init__()
        self.hidden = nn.Linear(d_in, d_hidden)
        self.act = nn.ReLU()
        self.out = nn.Linear(d_hidden, n_classes)

    def forward(self, x):
        return self.out(self.act(self.hidden(x)))     # logits

torch.manual_seed(0)
model = MLP(2, 64, 3)
print(model)
print("parameters:", sum(p.numel() for p in model.parameters()))
print("hidden weight shape:", model.hidden.weight.shape)

In [ ]:
model_seq = nn.Sequential(nn.Linear(2, 64), nn.ReLU(), nn.Linear(64, 3))
print(model_seq(torch.randn(5, 2)).shape)

In [ ]:
layer = nn.Linear(2, 3)
with torch.no_grad():
    layer.weight.copy_(torch.tensor([[1.0, 2.0], [0.0, -1.0], [3.0, 1.0]]))
    layer.bias.copy_(torch.tensor([0.0, 1.0, -2.0]))
print("layer output:", layer(torch.tensor([2.0, 1.0])).tolist())

loss_fn = nn.CrossEntropyLoss()
logits = torch.tensor([[2.0, 1.0, 0.1]])
target = torch.tensor([0])
print(f"correct: {loss_fn(logits, target).item():.3f}")
print(f"softmax applied twice (wrong): {loss_fn(torch.softmax(logits, dim=1), target).item():.3f}")

<div dir="rtl">

## الخسارة والمحسِّن

</div>

In [ ]:
logits = model(xb)              # 1. forward
loss = loss_fn(logits, yb)      # 2. loss
optimizer.zero_grad()           #    clear old gradients
loss.backward()                 # 3. backward: autograd
optimizer.step()                # 4. update every parameter

<div dir="rtl">

## التدريب الكامل: حلزون الدرس 37

</div>

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

def make_spiral(n_per_class=100, n_classes=3, noise=0.2, seed=0):
    rng = np.random.default_rng(seed)
    X, y = [], []
    for k in range(n_classes):
        r = np.linspace(0.0, 1.0, n_per_class)
        t = np.linspace(4 * k, 4 * (k + 1), n_per_class) + rng.normal(0, noise, n_per_class)
        X.append(np.column_stack([r * np.sin(t), r * np.cos(t)]))
        y.append(np.full(n_per_class, k))
    return np.vstack(X), np.concatenate(y)

X, y = make_spiral()
idx = np.random.default_rng(1).permutation(len(X))
X_tr = torch.tensor(X[idx[:240]], dtype=torch.float32)
y_tr = torch.tensor(y[idx[:240]], dtype=torch.long)       # class indices: integers
X_val = torch.tensor(X[idx[240:]], dtype=torch.float32)
y_val = torch.tensor(y[idx[240:]], dtype=torch.long)

torch.manual_seed(0)
model = MLP(2, 64, 3)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)
loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=32, shuffle=True)

def accuracy(model, X, y):
    model.eval()
    with torch.no_grad():
        return (model(X).argmax(dim=1) == y).float().mean().item()

for epoch in range(1, 201):
    model.train()
    for xb, yb in loader:
        loss = loss_fn(model(xb), yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    if epoch % 50 == 0:
        with torch.no_grad():
            val_loss = loss_fn(model(X_val), y_val).item()
        print(f"epoch {epoch}: val loss {val_loss:.3f}, val acc {accuracy(model, X_val, y_val):.3f}")

<div dir="rtl">

## الحفظ والتحميل

</div>

In [ ]:
print(list(model.state_dict().keys()))
torch.save(model.state_dict(), "spiral_mlp.pt")

restored = MLP(2, 64, 3)
restored.load_state_dict(torch.load("spiral_mlp.pt"))
print("same predictions:", torch.equal(restored(X_val).argmax(1), model(X_val).argmax(1)))

<div dir="rtl">

## كرت الشاشة

</div>

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

model = model.to(device)
xb = X_val.to(device)
print(model(xb).device)

In [ ]:
layer(torch.from_numpy(np.array([[2.0, 1.0]])))

<div dir="rtl">

## تمرين: تحقق من الدرس 36 بالتدرّج التلقائي

في الدرس 36 حسبنا بيدك تدرّجات شبكة صغيرة: مدخل 2، وهدف 1، والقيم الابتدائية $w = (0.5, -1)$، و $b = (0, 1)$، و $v = (1, 2)$، و $c = 0.5$.

1. أنشئ هذه القيم كموترات مع تسجيل التدرّج.
2. احسب الخسارة $(\hat{y} - y)^2$، واستدعِ الانتشار العكسي. تحقق أن التدرّجات: $w$: $(2, 0)$، و $b$: $(1, 0)$، و $v$: $(1, 0)$، و $c$: 1.
3. خذ خطوة بمحسِّن النزول بالتدرّج، بمعدل 0.1، وتحقق أن الخسارة الجديدة 0.0225.

</div>

In [ ]:
w = torch.tensor([0.5, -1.0], requires_grad=True)
# أكمل: b و v و c، ثم الخسارة، ثم backward